In [1]:
import pandas as pd

tabular = pd.read_csv("data/train/train_tabular.csv")
notes = pd.read_csv("data/train/train_notes.csv")

merged = pd.merge(
    tabular,
    notes,
    how="inner",
    on="flight_id",
)

merged["maintenance_note"] = merged["maintenance_note"].fillna("").astype(str)
merged["failure_mode"] = merged["failure_mode"].fillna("none").astype(str)


In [2]:
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np


class MaintenanceNoteTargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, smoothing=0.0):
        self.smoothing = smoothing

    def fit(self, X, y=None):
        X = X.copy()
        if isinstance(X, pd.DataFrame):
            X = X.iloc[:, 0]
        else:
            X = pd.Series(X[:, 0])

        y = np.asarray(y)
        self.global_mean_ = float(np.nanmean(y)) if y.size else 0.5
        self.note_means_ = (
            pd.Series(y, index=X)
            .groupby(level=0)
            .mean()
            .to_dict()
        )
        self.note_counts_ = (
            pd.Series(y, index=X)
            .groupby(level=0)
            .size()
            .to_dict()
        )
        return self

    def transform(self, X):
        X = X.copy()
        if isinstance(X, pd.DataFrame):
            values = X.iloc[:, 0]
        else:
            values = X[:, 0]

        encoded = []
        for value in values:
            if pd.isna(value) or value == "":
                encoded.append(self.global_mean_)
                continue
            count = self.note_counts_.get(value, 0)
            mean = self.note_means_.get(value, self.global_mean_)
            if self.smoothing > 0 and count > 0:
                encoded.append((count * mean + self.smoothing * self.global_mean_) / (count + self.smoothing))
            else:
                encoded.append(mean)
        return np.array(encoded, dtype=float).reshape(-1, 1)

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = ["maintenance_note"]
        input_features = np.atleast_1d(input_features)
        return np.array([f"{feature}_encoded" for feature in input_features], dtype=object)


In [3]:
X = merged.drop(columns=["flight_id", "drone_id", "failure_within_horizon", "failure_mode"])
y = merged["failure_within_horizon"]
g = merged["drone_id"]


In [4]:
categorical_columns = ["model", "motor_type", "firmware_version", "operator_region"]
numerical_columns = X.select_dtypes(include="number").columns
text_column = "maintenance_note"
aux_column = "failure_mode"


In [5]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_columns),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
        ("text", MaintenanceNoteTargetEncoder(smoothing=10), [text_column]),
    ],
    remainder="drop",
)


In [6]:
models = [
    LogisticRegression(max_iter=1000),
    RandomForestClassifier(n_estimators=100, random_state=42)
]

In [7]:
transformed = preprocessor.fit_transform(X, y)
transformed_dense = transformed.toarray() if hasattr(transformed, "toarray") else transformed
feature_names = preprocessor.get_feature_names_out()
transformed_df = pd.DataFrame(transformed_dense, columns=feature_names)

In [8]:
for model in models:
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model),
    ])

    scores = cross_val_score(
        pipeline,
        X,
        y,
        groups=g,
        cv=GroupKFold(n_splits=5),
        scoring="average_precision",
    )

    print(f"Model: {model}, Mean: {scores.mean()}, Std: {scores.std()}")


Model: LogisticRegression(max_iter=1000), Mean: 0.5271351855509675, Std: 0.029021119515271296
Model: RandomForestClassifier(random_state=42), Mean: 0.4977708514373084, Std: 0.03156350282814811


In [ ]:
from pathlib import Path
import joblib

artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(exist_ok=True)

candidate_models = [
    ("logreg", LogisticRegression(max_iter=3000, class_weight="balanced")),
    ("rf", RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")),
]

best_score = -np.inf
best_pipeline = None
best_name = None

for name, model in candidate_models:
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model),
    ])
    scores = cross_val_score(
        pipeline,
        X,
        y,
        groups=g,
        cv=GroupKFold(n_splits=5),
        scoring="average_precision",
    )
    print(f"{name}: mean AP={scores.mean():.4f}, std={scores.std():.4f}")
    if scores.mean() > best_score:
        best_score = scores.mean()
        best_pipeline = pipeline
        best_name = name

best_pipeline.fit(X, y)
joblib.dump(best_pipeline, artifacts_dir / "tabular_model.joblib")
joblib.dump(
    {"feature_columns": X.columns.tolist(), "target_column": "failure_within_horizon"},
    artifacts_dir / "tabular_model_metadata.joblib",
)
print(f"Saved best tabular model ({best_name}) to {artifacts_dir / 'tabular_model.joblib'}")